# Working with Datasets — User Guide

`StarLayerDataset` extends rdflib's `Dataset`: a single store holding several independently addressable named graphs, each an RDF-1.2-aware `StarLayerGraph`. 

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed. 

Run cells from top to bottom — later sections may reuse variables from earlier sections.

In [1]:
from starlayer import StarLayerDataset, StarLayerGraph, Namespace, TripleTerm, Literal

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Creating and populating named graphs

`dataset.graph(<name>)` returns the graph for that name.  If the graph does not exist it is created. 

In [2]:
ds = StarLayerDataset()
ds.bind("ex", EX)

g1 = ds.graph(EX.hr)
g2 = ds.graph(EX.crm)

g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))
g1.add((EX.alice, EX.title, Literal("Sales Manager")))
g2.add((EX.alice, EX.likes, EX.ProductABC))

print("g1 triple count:", len(g1))
print("g2 triple count:", len(g2))
print("graph names:", sorted(str(c.identifier) for c in ds.graphs()))

print("\nResulting dataset")
print(ds.serialize(format="trig12"))

g1 triple count: 2
g2 triple count: 1
graph names: ['http://example.org/crm', 'http://example.org/hr', 'urn:x-rdflib:default']

Resulting dataset
@prefix ex: <http://example.org/> .

GRAPH <http://example.org/hr> {
    ex:alice ex:title "Sales Manager" ;
        ex:worksFor ex:AcmeCorp .
}

GRAPH <http://example.org/crm> {
    ex:alice ex:likes ex:ProductABC .
}



## 2. Attaching statements to a graph's own IRI, and reifying within a named graph

Each named graph has an identifying IRI.  That IRI is just another node, so ordinary statements *about* the graph itself can be created.  Typically these statements would be added to the default graph or another named graph  of the dataset, since a graph can best be described outside itself.

Reification works the same way wherever the triple being reified lives — the example below reifies a claim living inside `ex:hr`, and separately reifies a claim *about* `ex:hr` itself, living in the dataset's default graph.

In [3]:
ds = StarLayerDataset()
ds.bind("ex", EX)

g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

# describe the ex:hr graph itself, in the dataset's default graph
ds.add((EX.hr, EX.maintainedBy, EX.DataTeam))

# reify a claim about a triple that lives inside the ex:hr graph
tt = TripleTerm(EX.alice, EX.worksFor, EX.AcmeCorp)
g1.add_reification(EX.claim, tt)
g1.add((EX.claim, EX.source, EX.HRSystem))

# reification works the same way on a triple that lives in the dataset's own
# default graph - ds.graph(<identifier>) returns a StarLayerGraph for any
# graph in the dataset, including the default graph itself.
default_graph = ds.graph(ds.default_graph.identifier)
graph_tt = TripleTerm(EX.hr, EX.maintainedBy, EX.DataTeam)
default_graph.add_reification(EX.hrMeta, graph_tt)
default_graph.add((EX.hrMeta, EX.lastUpdated, Literal("2026-09-16")))


print("\nResulting dataset")
print(ds.serialize(format="trig12"))

# note: the reifier and its annotation live inside GRAPH <ex:hr>, since that's the graph
# g1 refers to - while the statement about ex:hr itself, and its own reifier, live in
# the default graph.


Resulting dataset
@version "1.2" .

@prefix ex: <http://example.org/> .

GRAPH <urn:x-rdflib:default> {
    ex:hr ex:maintainedBy ex:DataTeam ~ ex:hrMeta {| ex:lastUpdated "2026-09-16" |} .
}

GRAPH <http://example.org/hr> {
    ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim {| ex:source ex:HRSystem |} .
}



### 2.1 Keeping annotations in a separate graph from the facts they annotate

Nothing requires a reifier and its annotations to live in the same graph as the triple they describe — `TripleTerm(s, p, o)` identifies a triple by value, not by which graph it's asserted in. This lets facts and their provenance/metadata be managed separately: one graph of raw facts, and a separate `annotation-graph` holding claims about those facts.

In [4]:
ds = StarLayerDataset()
ds.bind("ex", EX)

# the fact lives in ex:hr
g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

# the annotation about that same fact lives in a completely separate graph
ann = ds.graph(EX["annotation-graph"])
tt = TripleTerm(EX.alice, EX.worksFor, EX.AcmeCorp)
ann.add_reification(EX.claim, tt)
ann.add((EX.claim, EX.source, EX.HRSystem))

print("g1 triple count:", len(g1))
print("annotation-graph triple count:", len(ann))


print("\nResulting dataset")
print(ds.serialize(format="trig12"))

g1 triple count: 1
annotation-graph triple count: 2

Resulting dataset
@version "1.2" .

@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

GRAPH <http://example.org/annotation-graph> {
    ex:claim ex:source ex:HRSystem ;
        rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
}

GRAPH <http://example.org/hr> {
    ex:alice ex:worksFor ex:AcmeCorp .
}



### 2.2 Using reifier functions with datasets

`StarLayerDataset` also has dataset-wide versions of `reifiers()`/`reifications()`/`reifier_annotations()`/`reified_triples()`.  Each returns a new `StarLayerDataset` including the matching triples, still scoped to the named graph that contains the reifier.

In [5]:
ds = StarLayerDataset()
ds.bind("ex", EX)

# the fact lives in ex:hr
g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

# the annotation about that same fact lives in a separate graph
ann = ds.graph(EX["annotation-graph"])
tt = TripleTerm(EX.alice, EX.worksFor, EX.AcmeCorp)
ann.add_reification(EX.claim, tt)
ann.add((EX.claim, EX.source, EX.HRSystem))

# ds.reifiers(TT=tt) searches all graphs in the dataset at once 
result = ds.reifiers(TT=tt)

print("triples found:", len(result))
print("\nResulting result dataset")
print(result.serialize(format="trig12"))

print("\nResulting full dataset")
print(ds.serialize(format="trig12"))

triples found: 5

Resulting result dataset
@version "1.2" .

@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

GRAPH <http://example.org/annotation-graph> {
    ex:claim ex:source ex:HRSystem ;
        rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
}


Resulting full dataset
@version "1.2" .

@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

GRAPH <http://example.org/annotation-graph> {
    ex:claim ex:source ex:HRSystem ;
        rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
}

GRAPH <http://example.org/hr> {
    ex:alice ex:worksFor ex:AcmeCorp .
}



## 4. Serializing a dataset

A dataset can be serialized two ways, depending on whether graph boundaries need to be preserved:
- **4.1 TriG 1.2 and N-Quads 1.2** — graph-preserving: `trig12` uses one `GRAPH` block per named graph, `nq12` is one quad per line (`s p o g`).
- **4.2 Plain Turtle** — every named graph flattened into a single set of triples; graph boundaries are lost.

### 4.1 TriG 1.2 and N-Quads 1.2: graph-preserving serialization

Adding directly to the dataset (`ds.add(...)`, no graph specified) writes to the dataset's own default graph, separate from any named graph. Both `trig12` and `nq12` preserve graph boundaries — see the [serialization formats guide](05a-serialization-formats.ipynb) for their single-graph counterparts and the rest of the format set.

In [6]:
ds = StarLayerDataset()
ds.bind("ex", EX)

g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

g2 = ds.graph(EX.crm)
g2.add((EX.alice, EX.likes, EX.ProductABC))

ds.add((EX.hr, EX.maintainedBy, EX.DataTeam))
print("\nResulting dataset in trig12")
print(ds.serialize(format="trig12"))
print("\nResulting dataset in N-quads")
print(ds.serialize(format="nq12"))



Resulting dataset in trig12
@prefix ex: <http://example.org/> .

GRAPH <urn:x-rdflib:default> {
    ex:hr ex:maintainedBy ex:DataTeam .
}

GRAPH <http://example.org/hr> {
    ex:alice ex:worksFor ex:AcmeCorp .
}

GRAPH <http://example.org/crm> {
    ex:alice ex:likes ex:ProductABC .
}


Resulting dataset in N-quads
<http://example.org/hr> <http://example.org/maintainedBy> <http://example.org/DataTeam> <urn:x-rdflib:default> .
<http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> <http://example.org/hr> .
<http://example.org/alice> <http://example.org/likes> <http://example.org/ProductABC> <http://example.org/crm> .



### 4.2 Flattening a dataset into plain Turtle

Turtle has no syntax for multiple named graphs — plain Turtle is a single-graph format. `ds.serialize(format="turtle12")` (also `longturtle12`/`nt12`/`rdfxml12`/`jsonld12`) handles this automatically: every quad is flattened into one `StarLayerGraph` and serialized with that format, losing which graph each triple came from. `to_graph()` does the same flattening but returns the merged `StarLayerGraph` itself, for when the result needs to be queried or otherwise used as a graph rather than just printed.

In [7]:
ds = StarLayerDataset()
ds.bind("ex", EX)

g1 = ds.graph(EX.hr)
g1.add((EX.alice, EX.worksFor, EX.AcmeCorp))

g2 = ds.graph(EX.crm)
g2.add((EX.alice, EX.likes, EX.ProductABC))

ds.add((EX.hr, EX.maintainedBy, EX.DataTeam))

print(ds.serialize(format="turtle12"))

# to_graph() returns the merged StarLayerGraph itself, so it can be queried
# directly - not just printed
merged = ds.to_graph()
for row in merged.query("SELECT ?s WHERE { ?s ex:worksFor ?o }", initNs={"ex": EX}):
    print(row)

@prefix ex: <http://example.org/> .

ex:alice ex:likes ex:ProductABC ;
    ex:worksFor ex:AcmeCorp .

ex:hr ex:maintainedBy ex:DataTeam .

(rdflib.term.URIRef('http://example.org/alice'),)


## Further Reading

1. **[Getting Started](01-getting-started.ipynb)** — install, first parse, first query, first validate.
2. **[Graphs](02-graphs.ipynb)** — `TripleTerm`/`DirLangString` semantics, Turtle 1.2 reification syntax.
   - 2.a **Working with datasets** — this guide.
   - 2.b **[Inferencing](02b-graphs-inferencing.ipynb)** — RDFS and OWL 2 RL reasoning via `owlrl`.
3. **[SPARQL](03-sparql.ipynb)** — query semantics and built-in functions.
5. **Other**
   - 5.a **[Serialization formats](05a-serialization-formats.ipynb)** — all supported RDF 1.2 formats.
   - 5.b **[Working with backend graph databases](05b-backend-graph-databases.ipynb)** — blank-node handling on Oxigraph/Fuseki-backed datasets.
   - 5.d **[Canonical hashing and graph comparison](05d-canonical-hashing.ipynb)** — RDFC-1.0 canonicalization/hashing and graph isomorphism.